# 04 · True-talent xwOBA (empirical Bayes) — Phase 1

The product for *analyzing a hitter*: a per-batter-season xwOBA that is **regressed for sample
size** (a hot 120-PA line is pulled toward the league until the PAs earn it) with an interval
that **narrows with PA**. Method: Gaussian–Gaussian empirical Bayes, no BART re-fit —
`theta_hat = mu + reliability * (raw - mu)`, `reliability = tau^2 / (tau^2 + se^2)`.

Loads `results/talent/`.

In [ ]:
# --- setup: find repo root, load helpers (run this first) ---
import sys, json
from pathlib import Path
import polars as pl
from IPython.display import Image, Markdown, display

pl.Config.set_tbl_rows(30)
here = Path.cwd()
ROOT = next((p for p in [here, *here.parents] if (p / "config.yaml").exists()), here)
sys.path.insert(0, str(ROOT))
RESULTS = ROOT / "results"

def jload(rel):
    return json.loads((RESULTS / rel).read_text())

def show_fig(rel, width=680):
    p = RESULTS / rel
    return Image(filename=str(p), width=width) if p.exists() else Markdown(f"_missing figure: {rel}_")

print("repo root:", ROOT)
print("results:  ", RESULTS, "(exists)" if RESULTS.exists() else "(MISSING)")

## Per-season hyperparameters\nFit per season on players with 100+ PA, then applied to everyone.

In [ ]:
tm = jload("talent/talent_metrics.json")
pl.DataFrame([
    {"season": r["season"], "n": r["n"], "mu (league)": round(r["mu"], 4),
     "tau (spread)": round(r["tau"], 4), "median reliability": round(r["median_reliability"], 3)}
    for r in tm["per_season"]
])

## Shrinkage in action\nSmall samples collapse toward the season mean; big samples barely move.

In [ ]:
show_fig("talent/figures/shrinkage_raw_to_talent.png")

In [ ]:
show_fig("talent/figures/reliability_vs_pa.png")

## Concrete examples — raw → talent

In [ ]:
t = pl.read_parquet(RESULTS / "talent" / "talent_table.parquet")
def ex(name, season):
    r = t.filter((pl.col("player_name") == name) & (pl.col("season") == season)).row(0, named=True)
    return {"player": name, "season": season, "PA": r["PA"],
            "raw": round(r["xwoba_raw"], 3), "talent": round(r["xwoba_talent"], 3),
            "90% lo": round(r["talent_lo"], 3), "90% hi": round(r["talent_hi"], 3),
            "reliability": round(r["reliability"], 3)}
pl.DataFrame([
    ex("Khalil Lee", 2022),     # 2 PA, extreme hot -> fully regressed to league
    ex("Mike Trout", 2024),     # 125 PA, hot -> pulled down to a believable .341
    ex("Austin Hedges", 2024),  # 143 PA, cold, low-variance -> pulled up
])

Note Trout (boom-or-bust, high per-PA variance) is regressed *harder* than Hedges at
similar PA — the estimate uses each player's own noisiness, not just their PA count.

## The payoff — does shrinkage predict next season better than raw / Savant?

In [ ]:
v = tm["validation"]
def row(label, block, has_rmse):
    d = {"population": label, "n": block["n"],
         "EB talent r": round(block["xwoba_talent"]["r"], 4),
         "raw r": round(block["xwoba_raw"]["r"], 4),
         "Savant r": round(block["xwoba_savant"]["r"], 4)}
    return d
pl.DataFrame([
    row("pooled, PA_T >= 100 (vs 0.487 anchor)", v["pooled_pa_min"], True),
    row("pooled, PA_T >= 30 (admits low-PA)", v["pooled_lowpa_inclusive"], False),
])

**Verdict.** EB talent beats raw in both populations and **beats Savant once genuinely
low-PA seasons are admitted** (0.467 vs 0.452); at the 100+ PA anchor it's at parity with
Savant (0.489 vs 0.491) — the same tie as v0, but now with a sample-size-honest *center*.

### Why the win is *pooled*, not per-band (and not a bug)

In [ ]:
bands = pl.DataFrame([
    {"PA band": b["band"], "n": b["n"], "median reliability": round(b["median_reliability"], 2),
     "|talent-raw|": round(b["mean_abs_talent_minus_raw"], 4),
     "talent r": round(b["xwoba_talent"]["r"], 3), "raw r": round(b["xwoba_raw"]["r"], 3),
     "savant r": round(b["xwoba_savant"]["r"], 3)}
    for b in v["by_band"]
])
bands

Inside a narrow PA band, reliability is ~constant, so `theta_hat = mu(1-c) + c*raw` is
~**affine** in raw — and Pearson r is invariant to affine transforms. So per-band r *can't*
move much regardless of how correct the shrinkage is; the tiny per-band gaps are noise.
Shrinkage's real payoff is **variance-compression across a heterogeneous-PA population**
(taming wild low-PA raws), which shows up in the **pooled** number. Not a bug — a property of
the estimator, documented in `results/talent/NOTES.md`.

## Biggest shrinks (hot/cold small samples pulled to the mean)

In [ ]:
pl.DataFrame([
    {"player": r["player_name"], "season": r["season"], "PA": r["PA"],
     "raw": round(r["xwoba_raw"], 3), "talent": round(r["xwoba_talent"], 3),
     "reliability": round(r["reliability"], 3)}
    for r in tm["biggest_shrinks"][:10]
])

**Next: Phase 2.** Right now every player is shrunk toward the *league mean*. Phase 2
replaces that with a **BART contact-quality prior**, so a rookie barreling the ball regresses
toward a slugger, not toward average. See
`docs/superpowers/plans/2026-07-18-xwobart-phase2-bart-prior.md`.